In [47]:
# %% [markdown]
# # Гибридный ретривер: BM25 → Bi-encoder

# %% Импорты и декоратор
import json
import time
import tracemalloc
from functools import wraps
from typing import List, Dict, Callable
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

def measure_performance(func):
    """Кастомный декоратор: время + память"""
    @wraps(func)
    def wrapper(*args, **kwargs):
        tracemalloc.start()
        start = time.perf_counter()

        result = func(*args, **kwargs)

        elapsed = time.perf_counter() - start
        current, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()

        print(f"⏱ [{func.__name__}] Time: {elapsed:.3f}s | Mem: {current/1024**2:.1f}MB (peak: {peak/1024**2:.1f}MB)")
        return result
    return wrapper

# %% Универсальная функция извлечения текста
def extract_text(doc: dict, mode: str = 'full') -> str:
    """
    Универсальный экстрактор текста для разных форматов документов.

    Поддерживаемые форматы:
    1. {"heading": "...", "content": ["...", "..."]}  - плоский формат
    2. {"title": "...", "content": "...", "sections": [...]}  - вложенный формат
    """

    parts = []

    # Определяем формат и извлекаем данные
    if 'heading' in doc:
        # Новый плоский формат
        heading = doc.get('heading', '')
        content = doc.get('content', [])

        if mode == 'full':
            parts.append(heading)
            if isinstance(content, list):
                parts.extend(content)
            else:
                parts.append(str(content))
        elif mode == 'title_headings':
            parts.append(heading)
        elif mode == 'title_only':
            parts.append(heading)

    elif 'title' in doc:
        # Старый вложенный формат
        if mode == 'full':
            parts.append(doc.get('title', ''))
            parts.append(doc.get('content', ''))
            for section in doc.get('sections', []):
                parts.append(section.get('heading', ''))
                section_content = section.get('content', [])
                if isinstance(section_content, list):
                    parts.extend(section_content)
                else:
                    parts.append(str(section_content))
        elif mode == 'title_headings':
            parts.append(doc.get('title', ''))
            for section in doc.get('sections', []):
                parts.append(section.get('heading', ''))
        elif mode == 'title_only':
            parts.append(doc.get('title', ''))

    # Фильтруем пустые строки и объединяем
    return ' '.join(filter(None, parts))

# %% Стратегии преобразования документа → текст
STRATEGIES = {
    'full': lambda d: extract_text(d, 'full'),
    'title_headings': lambda d: extract_text(d, 'title_headings'),
    'title_only': lambda d: extract_text(d, 'title_only'),
}

# %% Функция получения заголовка документа
def get_doc_title(doc: dict) -> str:
    """Возвращает заголовок документа независимо от формата"""
    return doc.get('heading') or doc.get('title') or 'Без заголовка'

# %% Класс ретривера
class HybridRetriever:
    def __init__(
        self,
        docs: List[dict],
        model_name: str = 'intfloat/multilingual-e5-small',
        strategy: str = 'full'
    ):
        self.docs = docs
        self.doc_to_text = STRATEGIES[strategy]
        self.texts = [self.doc_to_text(d) for d in docs]

        # Отладка: покажем первые тексты
        print(f"📄 Загружено документов: {len(docs)}")
        for i, text in enumerate(self.texts[:3]):
            preview = text[:100] + "..." if len(text) > 100 else text
            print(f"   [{i}] {preview}")

        # BM25
        tokenized = [t.lower().split() for t in self.texts]
        self.bm25 = BM25Okapi(tokenized)

        # Bi-encoder
        self.model = SentenceTransformer(model_name)
        self.embeddings = None

    @measure_performance
    def encode_corpus(self):
        self.embeddings = self.model.encode(
            self.texts,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True
        )

    @measure_performance
    def search(self, query: str, bm25_k: int = 1000, top_k: int = 100) -> List[int]:
        # Защита от слишком большого bm25_k
        bm25_k = min(bm25_k, len(self.docs))
        top_k = min(top_k, bm25_k)

        # Stage 1: BM25
        scores_bm25 = self.bm25.get_scores(query.lower().split())
        candidates = np.argsort(scores_bm25)[::-1][:bm25_k]

        # Stage 2: Bi-encoder rerank
        q_emb = self.model.encode([query], normalize_embeddings=True)
        scores = self.embeddings[candidates] @ q_emb.T
        reranked = candidates[np.argsort(scores.flatten())[::-1][:top_k]]

        return reranked.tolist()

# %% Метрики
def compute_metrics(retrieved: List[int], relevant: set, ks: List[int]) -> dict:
    results = {}
    for k in ks:
        hits = len(set(retrieved[:k]) & relevant)
        results[f'P@{k}'] = hits / k
        results[f'R@{k}'] = hits / len(relevant) if relevant else 0
    return results

def evaluate(retriever, test_data: List[dict], ks=[1,3,5,8]) -> dict:
    all_metrics = {f'{m}@{k}': [] for m in ['P','R'] for k in ks}

    for item in test_data:
        retrieved = retriever.search(item['query'])
        relevant = set(item['relevant_ids'])

        for k in ks:
            hits = len(set(retrieved[:k]) & relevant)
            all_metrics[f'P@{k}'].append(hits / k)
            all_metrics[f'R@{k}'].append(hits / len(relevant) if relevant else 0)

    return {m: np.mean(v) for m, v in all_metrics.items()}

# %% Загрузка данных - НОВЫЙ ФОРМАТ'


import os

folder_path = '/content/drive/MyDrive/tbank_knowledge/'

# Список нужных индексов файлов
needed_indices = np.arange(1, 200000)

# Получаем все файлы из папки
all_files = sorted([f for f in os.listdir(folder_path) if f.endswith('.json')])

# Выбираем файлы по индексам
selected_files = [all_files[i] for i in needed_indices if i < len(all_files)]

# Список для всех данных
sample_docs = []

# Читаем выбранные файлы и объединяем
for filename in selected_files:
    file_path = os.path.join(folder_path, filename)

    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        # Если данные - список, добавляем все элементы
        if isinstance(data, list):
            sample_docs.extend(data)
            print(f"Прочитан {filename}: добавлено {len(data)} записей")
        else:
            # Если это один объект, добавляем его
            sample_docs.append(data)
            print(f"Прочитан {filename}: добавлена 1 запись")



#sample_docs = [
#     {
#         "heading": "Кто такой эмитент?",
#         "content": [
#             "Это юридическое лицо, которое выпускает в обращение акции и облигации, а потом продает их на бирже, чтобы получить дополнительное финансирование и использовать его для развития своего бизнеса.",
#             "Обычно выделяют три вида эмитентов.",
#             "Государство— в первую очередь оно выступает эмитентом национальной валюты, то есть всех денег в стране.",
#             "Органы муниципальной власти— они тоже могут выпускать облигации.",
#             "Частные компании—выпускают акции и облигации."
#         ]
#     },
#     {
#         "heading": "Кто такой депозитарий?",
#         "content": [
#             "Депозитарий — это профессиональный участник рынка ценных бумаг, который занимается учетом и хранением прав на ценные бумаги.",
#             "Депозитарий точно знает, какие ценные бумаги и в каком количестве принадлежат конкретному инвестору.",
#             "В Т‑Инвестициях есть собственный депозитарий, который получил лицензию на ведение деятельности от Центробанка России."
#         ]
#     },
#     {
#         "heading": "Что такое брокерский счет?",
#         "content": [
#             "Брокерский счет — это специальный счет для покупки и продажи ценных бумаг.",
#             "Открыть его можно у брокера, который имеет лицензию ЦБ."
#         ]
#     }
# ]

documents = sample_docs

# %% Эксперимент
CONFIG = {
    'model': 'intfloat/multilingual-e5-small',
    'strategy': 'full',
    'bm25_k': 50,  # Уменьшено для маленького корпуса
    'top_k': 10
}

print(f"Config: {CONFIG}\n")

# Инициализация
retriever = HybridRetriever(
    docs=documents,
    model_name=CONFIG['model'],
    strategy=CONFIG['strategy']
)
retriever.encode_corpus()

# %% Тестовый поиск
query = "Как перекинуть деньги со счёта у брокера на мой накопительный счёт?"
results = retriever.search(query, bm25_k=CONFIG['bm25_k'], top_k=CONFIG['top_k'])

print(f"\n🔍 Query: '{query}'")
print("=" * 50)
for i, idx in enumerate(results[:5], 1):
    title = get_doc_title(documents[idx])
    print(f"{i}. [{idx}] {title}")

# Корректируем k для маленького датасета
small_ks = [1, 2, 3]
metrics = evaluate(retriever, test_queries, ks=small_ks)

print("\n📊 Метрики качества:")
print("-" * 30)
for k in small_ks:
    print(f"k={k}: Precision={metrics[f'P@{k}']:.3f} | Recall={metrics[f'R@{k}']:.3f}")

# %% Дополнительные тесты
print("\n🧪 Дополнительные тестовые запросы:")
test_queries_extra = [
    "Как купить акцию?",
    "Как называется сумма, которую я получаяю по долговой ценной бумаге каждый месяц?"
]

for q in test_queries_extra:
    results = retriever.search(q, bm25_k=10, top_k=3)
    print(f"\n❓ {q}")
    for i, idx in enumerate(results, 1):
        print(f"   {i}. {get_doc_title(documents[idx])}")

Прочитан брокерский_счет.json: добавлена 1 запись
Прочитан ветер_перемен_что_делать_инвестору.json: добавлена 1 запись
Прочитан виды_бирж.json: добавлена 1 запись
Прочитан виртуальные_подарочные_акции_от_тинвестиций.json: добавлена 1 запись
Прочитан внебиржевые_торги_заблокированными_бумагами_с_хранением_в_нрд.json: добавлена 1 запись
Прочитан во_что_можно_инвестировать.json: добавлена 1 запись
Прочитан дивидендный_доход_по_акциям.json: добавлена 1 запись
Прочитан инвестиционные_счета.json: добавлена 1 запись
Прочитан как_вывести_деньги_с_брокерского_счета.json: добавлена 1 запись
Прочитан как_выставлять_биржевые_заявки.json: добавлена 1 запись
Прочитан как_заработать_на_акциях.json: добавлена 1 запись
Прочитан как_заработать_на_облигациях.json: добавлена 1 запись
Прочитан как_начать_торговать_на_бирже.json: добавлена 1 запись
Прочитан как_открыть_брокерский_счет.json: добавлена 1 запись
Прочитан как_покупать_и_продавать_драгоценные_металлы.json: добавлена 1 запись
Прочитан как_пок

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

⏱ [encode_corpus] Time: 35.190s | Mem: 0.1MB (peak: 0.9MB)
⏱ [search] Time: 0.108s | Mem: 0.0MB (peak: 0.1MB)

🔍 Query: 'Как перекинуть деньги со счёта у брокера на мой накопительный счёт?'
1. [8] Как вывести деньги с брокерского счета
2. [26] Нужно ли закрывать брокерский счет
3. [17] Как пополнить брокерский счет
4. [13] Как открыть брокерский счет
5. [45] Что такое брокерский счет
⏱ [search] Time: 0.089s | Mem: 0.0MB (peak: 0.1MB)
⏱ [search] Time: 0.093s | Mem: 0.0MB (peak: 0.1MB)
⏱ [search] Time: 0.097s | Mem: 0.0MB (peak: 0.1MB)
⏱ [search] Time: 0.088s | Mem: 0.0MB (peak: 0.1MB)

📊 Метрики качества:
------------------------------
k=1: Precision=0.250 | Recall=0.250
k=2: Precision=0.125 | Recall=0.250
k=3: Precision=0.083 | Recall=0.250

🧪 Дополнительные тестовые запросы:
⏱ [search] Time: 0.092s | Mem: 0.0MB (peak: 0.0MB)

❓ Как купить акцию?
   1. Как покупать и продавать ценные бумаги и валюту
   2. Как покупать и продавать драгоценные металлы
   3. Как начать торговать на бирже
